# Notebook 5: Fine-Tuning Best Practices

---

## Learning Objectives

By the end of this notebook, you will:

- Understand the difference between **fine-tuning** and **feature extraction**
- Know when to use each strategy based on dataset size and domain similarity
- Implement **gradual unfreezing** of pretrained layers
- Apply **discriminative learning rates** (different LR per layer group)
- Compare training strategies empirically and choose the best one
- Follow a best-practices checklist for transfer-learning projects

## Prerequisites

- **DL300 Notebooks 01-04**: CNN fundamentals, pooling, pretrained models, data augmentation
- **PyTorch**: Comfortable with `torch.nn`, `torch.optim`, dataloaders
- **Transfer learning basics**: Loading a pretrained model and replacing the classifier head

## Table of Contents

1. [Setup & Imports](#1.-Setup-&-Imports)
2. [Fine-Tuning vs Feature Extraction](#2.-Fine-Tuning-vs-Feature-Extraction)
3. [When to Fine-Tune](#3.-When-to-Fine-Tune)
4. [Gradual Unfreezing Strategy](#4.-Gradual-Unfreezing-Strategy)
5. [Learning Rates for Fine-Tuning](#5.-Learning-Rates-for-Fine-Tuning)
6. [Discriminative Learning Rates](#6.-Discriminative-Learning-Rates)
7. [Code: Progressive Unfreezing of ResNet](#7.-Code:-Progressive-Unfreezing-of-ResNet)
8. [Code: Compare Training Strategies](#8.-Code:-Compare-Training-Strategies)
9. [Best Practices Checklist](#9.-Best-Practices-Checklist)
10. [Common Mistakes & Debugging Tips](#10.-Common-Mistakes-&-Debugging-Tips)
11. [Exercises](#11.-Exercises)
12. [Exercise Solutions](#12.-Exercise-Solutions)

---

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import copy
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---

## 2. Fine-Tuning vs Feature Extraction

Transfer learning has two main flavors:

| Approach | What changes | When to use |
|----------|-------------|-------------|
| **Feature extraction** | Only the new classifier head trains; backbone weights are **frozen** | Small dataset, similar domain |
| **Fine-tuning** | Some or all backbone layers are **unfrozen** and trained with the head | Larger dataset, or domain differs from ImageNet |

### Feature Extraction

- Treat the pretrained CNN as a **fixed feature extractor**
- Replace the final classification layer with a new one matching your number of classes
- Only the new layer's weights are updated via backpropagation
- Fast to train; low risk of overfitting

### Fine-Tuning

- Start from a pretrained model, replace the head, **then also update backbone weights**
- The pretrained features are **adapted** to your specific task
- Higher capacity to learn task-specific features, but higher risk of overfitting on small data

**Key insight**: Early layers learn generic features (edges, textures) that transfer well. Later layers learn task-specific features that may need adaptation.

$$\text{Total loss} = \mathcal{L}(f_{\theta_{\text{backbone}}, \theta_{\text{head}}}(x), y)$$

- Feature extraction: only $\theta_{\text{head}}$ is updated
- Fine-tuning: both $\theta_{\text{backbone}}$ and $\theta_{\text{head}}$ are updated

---

## 3. When to Fine-Tune

The decision depends on **two factors**:

1. **Dataset size** (small vs large)
2. **Domain similarity** (similar to ImageNet vs very different)

| | Similar domain | Different domain |
|---|---|---|
| **Small dataset** | Feature extraction (freeze backbone) | Fine-tune carefully (risk of overfitting) |
| **Large dataset** | Fine-tune (can afford to update more) | Fine-tune aggressively (the more data, the safer) |

### Rules of Thumb

- **< 1,000 images per class**: Start with feature extraction. Fine-tune only the last 1-2 blocks if results are poor.
- **1,000 - 10,000 images per class**: Fine-tune the top layers (last 1-2 residual blocks).
- **> 10,000 images per class**: Fine-tune the entire network with a small learning rate.
- **Very different domain** (e.g., medical imaging, satellite): You may need to fine-tune more layers even with smaller datasets, because ImageNet features may not transfer well.

### The Danger Zone

- Small dataset + different domain = hardest case
- Use heavy data augmentation, aggressive regularization, and gradual unfreezing
- Consider finding a pretrained model closer to your domain

---

## 4. Gradual Unfreezing Strategy

Instead of unfreezing all layers at once, **gradually unfreeze** from the last layer backward:

1. **Phase 1**: Freeze all backbone layers. Train only the new classifier head for a few epochs.
2. **Phase 2**: Unfreeze the last residual block (or last convolutional group). Train for more epochs.
3. **Phase 3**: Unfreeze the second-to-last block. Continue training.
4. **Phase 4** (optional): Unfreeze the entire network.

**Why this works:**

- The classifier head starts with random weights. Training it first lets it "catch up" to the pretrained backbone.
- Unfreezing later layers first is safer because they encode more task-specific features.
- Early layers encode generic features (edges, corners) that rarely need updating.

```
ResNet-18 layer structure:

  [conv1] -> [bn1] -> [relu] -> [maxpool]   <-- Generic features (freeze longest)
       |                                       
  [layer1] (BasicBlock x2)                    <-- Low-level features
       |                                       
  [layer2] (BasicBlock x2)                    <-- Mid-level features
       |                                       
  [layer3] (BasicBlock x2)                    <-- Higher-level features
       |                                       
  [layer4] (BasicBlock x2)                    <-- Task-specific features (unfreeze first)
       |                                       
  [avgpool] -> [fc]                           <-- Classifier head (always train)
```

---

## 5. Learning Rates for Fine-Tuning

### Key Principle: Use a Smaller LR for Fine-Tuning

- Training from scratch: typical LR = $10^{-2}$ to $10^{-3}$
- Fine-tuning: typical LR = $10^{-4}$ to $10^{-5}$

**Why?** Pretrained weights are already in a good region of the loss landscape. A large learning rate would destroy these learned features.

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta} \mathcal{L}$$

If $\eta$ is too large, the update $\eta \cdot \nabla_{\theta} \mathcal{L}$ can push $\theta$ out of the good region, causing **catastrophic forgetting**.

### Practical Guidelines

- **Classifier head**: LR = $10^{-3}$ (new weights, needs bigger updates)
- **Last backbone block**: LR = $10^{-4}$ (pretrained, needs gentle updates)
- **Earlier backbone blocks**: LR = $10^{-5}$ or even smaller
- Use a **learning rate scheduler** (e.g., cosine annealing, reduce on plateau)

---

## 6. Discriminative Learning Rates

**Discriminative learning rates** (also called **differential learning rates** or **layer-wise LR decay**) assign **different learning rates to different layer groups**.

The idea: earlier layers need smaller updates (they already capture universal features), while later layers need larger updates (they must adapt to the new task).

### Common Pattern

Divide the network into $N$ groups and apply a decay factor $\gamma$ from the head backward:

$$\eta_i = \eta_{\text{base}} \cdot \gamma^{N - i}$$

where $i$ is the group index (0 = earliest layers, $N-1$ = head), and $\gamma \in (0, 1)$ is typically 0.1 or 0.3.

**Example with 4 groups and $\gamma = 0.1$, $\eta_{\text{base}} = 10^{-3}$:**

| Group | Layers | LR |
|-------|--------|----|
| 0 | conv1, layer1 | $10^{-3} \times 0.1^3 = 10^{-6}$ |
| 1 | layer2 | $10^{-3} \times 0.1^2 = 10^{-5}$ |
| 2 | layer3, layer4 | $10^{-3} \times 0.1^1 = 10^{-4}$ |
| 3 | fc (head) | $10^{-3} \times 0.1^0 = 10^{-3}$ |

### Implementation in PyTorch

PyTorch optimizers accept **parameter groups** with individual learning rates:

```python
optimizer = optim.Adam([
    {"params": early_params,  "lr": 1e-6},
    {"params": mid_params,    "lr": 1e-5},
    {"params": late_params,   "lr": 1e-4},
    {"params": head_params,   "lr": 1e-3},
])
```

---

## 7. Code: Progressive Unfreezing of ResNet

Let us implement the gradual unfreezing strategy on a ResNet-18 model.

In [ ]:
def create_resnet18_for_transfer(num_classes: int = 10, pretrained: bool = True):
    """Create a ResNet-18 with a new classifier head.
    
    Args:
        num_classes: Number of output classes.
        pretrained: Whether to load ImageNet pretrained weights.
    
    Returns:
        model: Modified ResNet-18.
    """
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    
    # Replace the classifier head
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, num_classes)
    )
    return model


# Create model
model = create_resnet18_for_transfer(num_classes=10)
print(f"Model created. Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def freeze_all_layers(model):
    """Freeze all parameters in the model."""
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_layer_group(model, group_name: str):
    """Unfreeze a specific named group of layers.
    
    Args:
        model: The ResNet model.
        group_name: Name of the layer group (e.g., 'layer4', 'layer3', 'fc').
    """
    for name, param in model.named_parameters():
        if group_name in name:
            param.requires_grad = True


def count_trainable_params(model):
    """Count the number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def print_freeze_status(model):
    """Print which layer groups are frozen vs unfrozen."""
    groups = ['conv1', 'bn1', 'layer1', 'layer2', 'layer3', 'layer4', 'fc']
    for group in groups:
        params = [p for n, p in model.named_parameters() if group in n]
        if params:
            trainable = sum(p.requires_grad for p in params)
            total = len(params)
            status = "UNFROZEN" if trainable == total else ("FROZEN" if trainable == 0 else "PARTIAL")
            print(f"  {group:8s}: {status} ({trainable}/{total} param tensors trainable)")

In [ ]:
# Demonstrate progressive unfreezing

print("=" * 60)
print("Phase 1: Freeze everything, unfreeze only the classifier head")
print("=" * 60)
freeze_all_layers(model)
unfreeze_layer_group(model, 'fc')
print_freeze_status(model)
print(f"Trainable params: {count_trainable_params(model):,}\n")

print("=" * 60)
print("Phase 2: Also unfreeze layer4")
print("=" * 60)
unfreeze_layer_group(model, 'layer4')
print_freeze_status(model)
print(f"Trainable params: {count_trainable_params(model):,}\n")

print("=" * 60)
print("Phase 3: Also unfreeze layer3")
print("=" * 60)
unfreeze_layer_group(model, 'layer3')
print_freeze_status(model)
print(f"Trainable params: {count_trainable_params(model):,}\n")

print("=" * 60)
print("Phase 4: Unfreeze everything")
print("=" * 60)
for param in model.parameters():
    param.requires_grad = True
print_freeze_status(model)
print(f"Trainable params: {count_trainable_params(model):,}")

---

## 8. Code: Compare Training Strategies

We will compare three strategies on a small subset of CIFAR-10:

1. **Feature extraction**: Only train the head
2. **Full fine-tuning**: Unfreeze everything with a single LR
3. **Discriminative LR fine-tuning**: Different LR per layer group

We use a **small subset** (500 training images) to simulate a low-data scenario.

In [ ]:
# Prepare a small CIFAR-10 subset
set_global_seed(42)

# CIFAR-10 images are 32x32; ResNet expects 224x224, but we resize to 64x64
# for speed. In production, use 224x224 for best results.
transform_train = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_train = torchvision.datasets.CIFAR10(
    root='../../data', train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='../../data', train=False, download=True, transform=transform_test
)

# Take a small subset: 500 training images, 200 test images
train_indices = list(range(500))
test_indices = list(range(200))

train_subset = Subset(full_train, train_indices)
test_subset = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_subset)}")
print(f"Test samples:     {len(test_subset)}")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch. Returns average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate on a dataset. Returns average loss and accuracy."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / total, correct / total

In [ ]:
def run_strategy(strategy_name, model, optimizer, num_epochs=8):
    """Train a model with a given strategy and return history."""
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)
    
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name}")
    print(f"Trainable params: {count_trainable_params(model):,}")
    print(f"{'='*60}")
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        print(f"  Epoch {epoch+1:2d}/{num_epochs}: "
              f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.3f} | "
              f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.3f}")
    
    return history

In [ ]:
# --- Strategy 1: Feature Extraction (frozen backbone) ---
set_global_seed(42)
model_fe = create_resnet18_for_transfer(num_classes=10)
freeze_all_layers(model_fe)
unfreeze_layer_group(model_fe, 'fc')

optimizer_fe = optim.Adam(
    filter(lambda p: p.requires_grad, model_fe.parameters()),
    lr=1e-3
)

history_fe = run_strategy("Feature Extraction (frozen backbone)", model_fe, optimizer_fe)

In [ ]:
# --- Strategy 2: Full Fine-Tuning (single LR) ---
set_global_seed(42)
model_ft = create_resnet18_for_transfer(num_classes=10)
# All layers unfrozen (default)

optimizer_ft = optim.Adam(model_ft.parameters(), lr=1e-4)

history_ft = run_strategy("Full Fine-Tuning (single LR=1e-4)", model_ft, optimizer_ft)

In [ ]:
# --- Strategy 3: Discriminative Learning Rates ---
set_global_seed(42)
model_dlr = create_resnet18_for_transfer(num_classes=10)

# Define parameter groups with discriminative learning rates
param_groups = [
    # Group 0: Early layers (conv1, bn1, layer1) - smallest LR
    {
        'params': [p for n, p in model_dlr.named_parameters()
                   if any(prefix in n for prefix in ['conv1', 'bn1', 'layer1'])],
        'lr': 1e-6
    },
    # Group 1: layer2 - small LR
    {
        'params': [p for n, p in model_dlr.named_parameters() if 'layer2' in n],
        'lr': 1e-5
    },
    # Group 2: layer3 + layer4 - medium LR
    {
        'params': [p for n, p in model_dlr.named_parameters()
                   if any(prefix in n for prefix in ['layer3', 'layer4'])],
        'lr': 1e-4
    },
    # Group 3: classifier head - largest LR
    {
        'params': [p for n, p in model_dlr.named_parameters() if 'fc' in n],
        'lr': 1e-3
    },
]

optimizer_dlr = optim.Adam(param_groups)

print("Discriminative LR setup:")
for i, group in enumerate(optimizer_dlr.param_groups):
    n_params = sum(p.numel() for p in group['params'])
    print(f"  Group {i}: LR={group['lr']:.1e}, params={n_params:,}")

history_dlr = run_strategy("Discriminative LR Fine-Tuning", model_dlr, optimizer_dlr)

In [ ]:
# Plot comparison of all three strategies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

strategies = {
    'Feature Extraction': history_fe,
    'Full Fine-Tuning': history_ft,
    'Discriminative LR': history_dlr,
}

colors = ['#2196F3', '#FF5722', '#4CAF50']

for ax_idx, metric in enumerate(['test_acc', 'test_loss']):
    ax = axes[ax_idx]
    for (name, hist), color in zip(strategies.items(), colors):
        epochs = range(1, len(hist[metric]) + 1)
        ax.plot(epochs, hist[metric], marker='o', label=name, color=color, linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test Accuracy' if metric == 'test_acc' else 'Test Loss')
    ax.set_title('Test Accuracy Comparison' if metric == 'test_acc' else 'Test Loss Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final results
print("\nFinal Test Accuracies:")
print("-" * 45)
for name, hist in strategies.items():
    print(f"  {name:25s}: {hist['test_acc'][-1]:.3f}")

---

## 9. Best Practices Checklist

Use this checklist for every fine-tuning project:

### Before Training

- [ ] **Start with a good pretrained model** matching your input domain (e.g., ImageNet for natural images)
- [ ] **Match preprocessing**: Use the same normalization (mean, std) the pretrained model was trained with
- [ ] **Replace only the final classifier** head to match your number of classes
- [ ] **Use data augmentation** appropriate for your domain
- [ ] **Set random seeds** for reproducibility

### During Training

- [ ] **Start frozen**: Train only the head for a few epochs first
- [ ] **Use a small learning rate** for fine-tuning ($10^{-4}$ to $10^{-5}$ for backbone)
- [ ] **Use discriminative LRs**: Larger LR for head, smaller for backbone
- [ ] **Unfreeze gradually**: Last layers first, then progressively earlier layers
- [ ] **Monitor both train and validation metrics** to detect overfitting
- [ ] **Use a LR scheduler**: Cosine annealing or reduce-on-plateau
- [ ] **Apply regularization**: Dropout, weight decay, early stopping

### After Training

- [ ] **Compare strategies**: Test feature extraction vs fine-tuning on your data
- [ ] **Save the best model** (by validation metric, not training metric)
- [ ] **Document your choices**: Which layers were unfrozen, what LRs were used, etc.
- [ ] **Check for overfitting**: Large train-test gap means you may need more regularization

### Common Pitfalls to Avoid

- Using the same large LR for backbone and head
- Not matching the pretrained model's preprocessing (normalization)
- Fine-tuning with too little data and no augmentation
- Unfreezing all layers at once from epoch 1
- Forgetting to set `model.train()` / `model.eval()` (affects BatchNorm and Dropout)

---

## 10. Common Mistakes & Debugging Tips

### Mistake 1: Forgetting to freeze BatchNorm statistics

Even when you "freeze" layers by setting `requires_grad = False`, **BatchNorm** layers still update their running mean and variance during `model.train()` mode.

**Fix**: If you want truly frozen BatchNorm, either:
- Keep the model (or specific modules) in `eval()` mode
- Use a helper to freeze BN layers:

```python
def freeze_bn(module):
    if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        module.eval()
```

### Mistake 2: Not filtering parameters for the optimizer

If you freeze layers but pass `model.parameters()` to the optimizer, it will still track momentum/state for frozen params, wasting memory.

**Fix**: Filter trainable parameters:
```python
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)
```

### Mistake 3: Using mismatched normalization

Pretrained ImageNet models expect `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`. Using different normalization will produce garbage features.

### Mistake 4: Learning rate too high

**Symptom**: Loss spikes or diverges after unfreezing layers.

**Fix**: Reduce the LR by 10x when unfreezing. Use a warmup schedule.

### Mistake 5: Unfreezing all layers immediately with small data

**Symptom**: Training accuracy goes to 100% while validation accuracy drops.

**Fix**: Start with feature extraction, then gradually unfreeze.

### Debugging Checklist

1. **Print trainable parameter count** at each phase
2. **Check `requires_grad`** for a few representative parameters
3. **Monitor gradient norms** to detect vanishing or exploding gradients
4. **Verify preprocessing** matches the pretrained model expectations
5. **Test with a single batch** first to make sure the pipeline works

In [ ]:
# Debugging utility: check gradient flow in a model

def check_gradient_flow(model):
    """Print gradient statistics for each layer group.
    
    Run this AFTER a backward pass to inspect gradient health.
    """
    print(f"{'Layer':<30s} {'requires_grad':<15s} {'grad mean':<15s} {'grad std':<15s}")
    print("-" * 75)
    for name, param in model.named_parameters():
        if param.requires_grad:
            if param.grad is not None:
                print(f"{name:<30s} {str(param.requires_grad):<15s} "
                      f"{param.grad.mean().item():<15.2e} {param.grad.std().item():<15.2e}")
            else:
                print(f"{name:<30s} {str(param.requires_grad):<15s} {'No grad':<15s}")


# Quick demonstration
demo_model = create_resnet18_for_transfer(num_classes=10).to(device)
freeze_all_layers(demo_model)
unfreeze_layer_group(demo_model, 'fc')
unfreeze_layer_group(demo_model, 'layer4')

# Run one forward + backward pass
demo_input = torch.randn(4, 3, 64, 64).to(device)
demo_target = torch.tensor([0, 1, 2, 3]).to(device)
demo_output = demo_model(demo_input)
demo_loss = nn.CrossEntropyLoss()(demo_output, demo_target)
demo_loss.backward()

check_gradient_flow(demo_model)

del demo_model, demo_input, demo_target, demo_output, demo_loss

---

## 11. Exercises

### Exercise 1: Implement Gradual Unfreezing Training Loop

Write a function `train_with_gradual_unfreezing` that:
1. Trains only the head for 3 epochs
2. Unfreezes `layer4` and trains for 3 more epochs
3. Unfreezes `layer3` and trains for 3 more epochs
4. Returns the training history

Use a learning rate of `1e-3` for the head phase and `1e-4` after unfreezing.

### Exercise 2: Compare on Different Data Sizes

Using the strategies from Section 8, compare feature extraction vs full fine-tuning on:
- 100 training samples
- 1000 training samples
- 5000 training samples

Plot the final test accuracy vs dataset size for each strategy. Which strategy wins at each data size?

### Exercise 3: Implement Learning Rate Warmup

Implement a linear warmup schedule that:
- Starts at `lr=0` and linearly increases to the target LR over the first 2 epochs
- Then follows cosine annealing for the remaining epochs

Hint: use `torch.optim.lr_scheduler.SequentialLR` with `LinearLR` and `CosineAnnealingLR`.

---

## 12. Exercise Solutions

### Solution 1: Gradual Unfreezing Training Loop

In [ ]:
def train_with_gradual_unfreezing(num_classes=10, epochs_per_phase=3):
    """Train a ResNet-18 using gradual unfreezing.
    
    Phase 1: Head only (3 epochs, lr=1e-3)
    Phase 2: Head + layer4 (3 epochs, lr=1e-4)
    Phase 3: Head + layer4 + layer3 (3 epochs, lr=1e-4)
    
    Returns:
        history: Dict with train/test loss and accuracy.
    """
    set_global_seed(42)
    model = create_resnet18_for_transfer(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'phase': []}
    
    phases = [
        ('Phase 1: Head only',          ['fc'],                    1e-3),
        ('Phase 2: Head + layer4',      ['fc', 'layer4'],         1e-4),
        ('Phase 3: Head + layer4 + layer3', ['fc', 'layer4', 'layer3'], 1e-4),
    ]
    
    epoch_count = 0
    for phase_name, groups_to_unfreeze, lr in phases:
        print(f"\n--- {phase_name} (LR={lr}) ---")
        
        # Freeze all, then unfreeze the specified groups
        freeze_all_layers(model)
        for group in groups_to_unfreeze:
            unfreeze_layer_group(model, group)
        
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr
        )
        
        print(f"  Trainable params: {count_trainable_params(model):,}")
        
        for ep in range(epochs_per_phase):
            epoch_count += 1
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            test_loss, test_acc = evaluate(model, test_loader, criterion, device)
            
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['test_loss'].append(test_loss)
            history['test_acc'].append(test_acc)
            history['phase'].append(phase_name)
            
            print(f"  Epoch {epoch_count:2d}: "
                  f"Train Acc={train_acc:.3f}, Test Acc={test_acc:.3f}")
    
    return history


history_gradual = train_with_gradual_unfreezing()
print(f"\nFinal test accuracy: {history_gradual['test_acc'][-1]:.3f}")

### Solution 2: Compare on Different Data Sizes

In [ ]:
def run_experiment_for_size(n_train, num_epochs=6):
    """Run feature extraction and fine-tuning for a given training set size."""
    set_global_seed(42)
    
    # Create subset loaders
    train_sub = Subset(full_train, list(range(n_train)))
    loader_tr = DataLoader(train_sub, batch_size=32, shuffle=True, num_workers=0)
    
    results = {}
    
    for strategy_name, freeze_backbone in [('Feature Extraction', True), ('Fine-Tuning', False)]:
        set_global_seed(42)
        model = create_resnet18_for_transfer(num_classes=10).to(device)
        
        if freeze_backbone:
            freeze_all_layers(model)
            unfreeze_layer_group(model, 'fc')
            lr = 1e-3
        else:
            lr = 1e-4
        
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr
        )
        criterion = nn.CrossEntropyLoss()
        
        for epoch in range(num_epochs):
            train_one_epoch(model, loader_tr, criterion, optimizer, device)
        
        _, test_acc = evaluate(model, test_loader, criterion, device)
        results[strategy_name] = test_acc
        print(f"  n_train={n_train:5d}, {strategy_name:20s}: Test Acc={test_acc:.3f}")
    
    return results


data_sizes = [100, 1000, 5000]
all_results = {}

for size in data_sizes:
    all_results[size] = run_experiment_for_size(size)

# Plot results
fig, ax = plt.subplots(figsize=(8, 5))
for strategy in ['Feature Extraction', 'Fine-Tuning']:
    accs = [all_results[s][strategy] for s in data_sizes]
    ax.plot(data_sizes, accs, marker='o', linewidth=2, label=strategy)

ax.set_xlabel('Training Set Size')
ax.set_ylabel('Test Accuracy')
ax.set_title('Feature Extraction vs Fine-Tuning at Different Data Sizes')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Solution 3: Learning Rate Warmup

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

set_global_seed(42)

# Create model and optimizer
model_warmup = create_resnet18_for_transfer(num_classes=10).to(device)
optimizer_warmup = optim.Adam(model_warmup.parameters(), lr=1e-4)

# Define schedulers
total_epochs = 10
warmup_epochs = 2

warmup_scheduler = LinearLR(
    optimizer_warmup,
    start_factor=0.01,   # Start at 1% of target LR
    end_factor=1.0,      # Ramp up to 100%
    total_iters=warmup_epochs
)
cosine_scheduler = CosineAnnealingLR(
    optimizer_warmup,
    T_max=total_epochs - warmup_epochs
)
scheduler = SequentialLR(
    optimizer_warmup,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs]
)

# Track LR over epochs
lr_history = []
criterion = nn.CrossEntropyLoss()

for epoch in range(total_epochs):
    current_lr = optimizer_warmup.param_groups[0]['lr']
    lr_history.append(current_lr)
    
    train_loss, train_acc = train_one_epoch(
        model_warmup, train_loader, criterion, optimizer_warmup, device
    )
    test_loss, test_acc = evaluate(model_warmup, test_loader, criterion, device)
    
    scheduler.step()
    print(f"Epoch {epoch+1:2d}: LR={current_lr:.2e}, Test Acc={test_acc:.3f}")

# Plot LR schedule
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, total_epochs + 1), lr_history, marker='o', linewidth=2)
ax.axvline(x=warmup_epochs, color='r', linestyle='--', alpha=0.5, label='Warmup ends')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Linear Warmup + Cosine Annealing Schedule')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

**Next notebook**: [06 - Introduction to Detection and Segmentation Concepts](06_Intro_to_Detection_and_Segmentation_Concepts.ipynb)